# Seasonal Agriculture Performance Analysis

## VOIS Major Project

This notebook analyzes the provided seasonal agriculture performance dataset to identify patterns in agricultural yield, profitability, resource use, farming practices, environmental conditions, and disease/pest risk.

**Important:** The analysis is descriptive and correlational. Relationships identified in the data should not automatically be interpreted as causal effects.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import f_oneway, pearsonr

df = pd.read_csv("seasonal_agriculture_performance_dataset.csv")
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

print("Dataset shape:", df.shape)
df.head()


## 1. Data Understanding

The dataset contains farm-level observations with geographic, crop, seasonal, environmental, input, production, financial, water-use, and disease/pest variables.


In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

print("\nDuplicate rows:", df.duplicated().sum())

print("\nCategorical distributions:")
for col in ["state", "district", "crop", "season", "irrigation_method"]:
    print(f"\n{col}:")
    print(df[col].value_counts())


## 2. Data Cleaning

There are missing observations in rainfall, soil moisture, and yield. No duplicate rows were found.

For the analytical dataset, missing values in these three numerical variables are replaced with their respective medians. Median imputation is used because it is less sensitive to extreme values than mean imputation and retains the available farm records.

The original raw dataset is not overwritten.


In [ ]:
analysis_df = df.copy()

for col in ["rainfall_mm", "soil_moisture_pct", "yield_tonnes_ha"]:
    analysis_df[col] = analysis_df[col].fillna(analysis_df[col].median())

print("Remaining missing values:")
print(analysis_df.isna().sum()[analysis_df.isna().sum() > 0])


## 3. Seasonal Performance

We compare yield, revenue, cost, profit, water efficiency, and disease/pest risk across Kharif, Rabi, and Zaid.


In [ ]:
season = analysis_df.groupby("season").agg(
    farms=("farm_id","count"),
    avg_yield_t_ha=("yield_tonnes_ha","mean"),
    avg_revenue_inr=("revenue_inr","mean"),
    avg_cost_inr=("total_cost_inr","mean"),
    avg_profit_inr=("profit_inr","mean"),
    avg_water_efficiency=("water_efficiency_t_per_1000m3","mean"),
    avg_risk_pct=("disease_pest_risk_pct","mean")
).reindex(["Kharif","Rabi","Zaid"])

season.round(2)


In [ ]:
plt.figure(figsize=(8,5))
plt.bar(season.index, season["avg_yield_t_ha"])
plt.ylabel("Average Yield (tonnes/ha)")
plt.title("Average Yield by Season")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
plt.bar(season.index, season["avg_profit_inr"])
plt.axhline(0, linewidth=1)
plt.ylabel("Average Profit (INR)")
plt.title("Average Profit by Season")
plt.tight_layout()
plt.show()


### Seasonal interpretation

Kharif has the highest descriptive average yield and profit, while Zaid has the lowest average yield and negative average profit.

However, a one-way ANOVA should be used before claiming that season itself produces statistically significant differences in yield.


In [ ]:
groups = [g["yield_tonnes_ha"].dropna() for _, g in analysis_df.groupby("season")]
anova = f_oneway(*groups)

print(f"ANOVA F-statistic: {anova.statistic:.3f}")
print(f"ANOVA p-value: {anova.pvalue:.6f}")

if anova.pvalue < 0.05:
    print("Conclusion: yield differences across seasons are statistically significant at the 5% level.")
else:
    print("Conclusion: the data do not provide statistically significant evidence of a seasonal yield difference at the 5% level.")


## 4. Crop Performance

Crop-level profitability helps identify which crops contribute most strongly to economic performance.


In [ ]:
crop = analysis_df.groupby("crop").agg(
    avg_yield_t_ha=("yield_tonnes_ha","mean"),
    avg_profit_inr=("profit_inr","mean"),
    avg_revenue_inr=("revenue_inr","mean")
).sort_values("avg_profit_inr", ascending=False)

crop.round(2)


In [ ]:
plt.figure(figsize=(9,5))
vals = crop["avg_profit_inr"].sort_values()
plt.barh(vals.index, vals.values)
plt.axvline(0, linewidth=1)
plt.xlabel("Average Profit (INR)")
plt.title("Average Profit by Crop")
plt.tight_layout()
plt.show()


## 5. State and District Performance

Geographic comparisons show where average agricultural yield and profitability are strongest in the supplied dataset.


In [ ]:
state = analysis_df.groupby("state").agg(
    avg_yield_t_ha=("yield_tonnes_ha","mean"),
    avg_profit_inr=("profit_inr","mean")
).sort_values("avg_yield_t_ha", ascending=False)

state.round(2)


In [ ]:
plt.figure(figsize=(9,5))
vals = state["avg_yield_t_ha"].sort_values()
plt.barh(vals.index, vals.values)
plt.xlabel("Average Yield (tonnes/ha)")
plt.title("Average Yield by State")
plt.tight_layout()
plt.show()


## 6. Irrigation Method

Irrigation methods are compared using average yield, profit, and water efficiency.


In [ ]:
irrigation = analysis_df.groupby("irrigation_method").agg(
    avg_yield_t_ha=("yield_tonnes_ha","mean"),
    avg_profit_inr=("profit_inr","mean"),
    avg_water_efficiency=("water_efficiency_t_per_1000m3","mean")
).sort_values("avg_yield_t_ha", ascending=False)

irrigation.round(2)


In [ ]:
plt.figure(figsize=(9,5))
vals = irrigation["avg_yield_t_ha"].sort_values()
plt.barh(vals.index, vals.values)
plt.xlabel("Average Yield (tonnes/ha)")
plt.title("Average Yield by Irrigation Method")
plt.tight_layout()
plt.show()


## 7. Water Efficiency

Water efficiency is examined in relation to yield. Pearson correlation measures the strength of the linear association between the two numerical variables.


In [ ]:
plot_df = analysis_df[["yield_tonnes_ha","water_efficiency_t_per_1000m3"]].dropna()

r, p = pearsonr(
    plot_df["water_efficiency_t_per_1000m3"],
    plot_df["yield_tonnes_ha"]
)

print(f"Pearson correlation (yield vs water efficiency): r = {r:.3f}")
print(f"p-value = {p:.6g}")


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(
    plot_df["water_efficiency_t_per_1000m3"],
    plot_df["yield_tonnes_ha"],
    alpha=0.35
)
plt.xlabel("Water Efficiency (tonnes per 1,000 m³)")
plt.ylabel("Yield (tonnes/ha)")
plt.title("Yield vs Water Efficiency")
plt.tight_layout()
plt.show()


## 8. Disease and Pest Risk

Disease/pest risk is compared across seasons to identify where risk levels are descriptively higher.


In [ ]:
risk = analysis_df.groupby("season")["disease_pest_risk_pct"].mean().reindex(
    ["Kharif","Rabi","Zaid"]
)

print(risk.round(2))

plt.figure(figsize=(8,5))
plt.bar(risk.index, risk.values)
plt.ylabel("Average Disease/Pest Risk (%)")
plt.title("Disease/Pest Risk by Season")
plt.tight_layout()
plt.show()


## 9. Correlation Analysis

Correlation analysis helps identify variables that move together with yield. Correlation does not establish causation.


In [ ]:
numeric_corr = analysis_df.select_dtypes(include=np.number).corr()["yield_tonnes_ha"].sort_values(ascending=False)
numeric_corr.round(3)


## 10. Key Findings

1. Kharif has the highest descriptive average yield and profit among the three seasons.
2. Zaid has the lowest descriptive average yield and negative average profit.
3. Sugarcane and Chilli have the strongest average profitability among the listed crops.
4. Punjab has the highest average yield among the states in this dataset.
5. Drip irrigation has the highest average yield and average profit among the listed irrigation methods.
6. Yield has a strong positive correlation with the dataset's water-efficiency measure.
7. The one-way ANOVA does not provide statistically significant evidence of a yield difference across seasons at the 5% level (`p ≈ 0.233`).

These findings describe the supplied sample and should not be generalized beyond it without additional evidence.


## 11. Recommendations

- Prioritize profitability analysis when selecting crop-season combinations rather than relying on yield alone.
- Investigate the conditions associated with the strong performance of Sugarcane and Chilli.
- Evaluate efficient irrigation practices, particularly drip irrigation, while considering crop and regional suitability.
- Pay attention to Zaid-season economics because its average profit is negative in the supplied data.
- Use water-efficiency indicators alongside yield when evaluating resource use.
- Use further field-level or longitudinal studies before making causal claims about environmental factors, irrigation, or seasonal effects.


## 12. Conclusion

The analysis shows meaningful variation in agricultural performance across seasons, crops, states, and irrigation methods. Kharif demonstrates the strongest descriptive economic performance, while Zaid presents a profitability concern. Crop choice and irrigation method are associated with substantial differences in observed performance.

The results provide a data-driven basis for comparing agricultural practices and identifying areas for further investigation, while statistical testing and correlation analysis help prevent overinterpretation of the observed patterns.
